In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from typing import Tuple


In [11]:
# Positive effect data
np.random.seed(42)
n = 1000
positive_effect_data = pd.DataFrame({
    "I": range(n),
    "T": np.random.binomial(1, 0.5, n),
})
positive_effect_data["Y"] = np.where(
    positive_effect_data["T"] == 1,
    np.random.normal(10, 2, n),
    np.random.normal(8, 2, n),
)

positive_effect_data.head()

,I,T,Y
0,0,0,5.187365
1,1,1,7.329311
2,2,1,10.760396
3,3,1,11.221171
4,4,0,8.164880


In [12]:
# No effect data
np.random.seed(123)
n = 500
no_effect_data = pd.DataFrame({
    "I": range(n),
    "T": np.random.binomial(1, 0.5, n),
})
no_effect_data["Y"] = np.random.normal(5, 3, n)

no_effect_data.head()

,I,T,Y
0,0,1,3.527083
1,1,0,1.072504
2,2,0,4.974019
3,3,1,7.930439
4,4,1,-0.253211


In [13]:
def calculate_ate_ci(data: pd.DataFrame, alpha: float = 0.05) -> Tuple[float, float, float]:
    """
    Calculate the Average Treatment Effect (ATE) and its confidence interval.

    Parameters:
    data (pd.DataFrame): A DataFrame containing columns 'T' (treatment) and 'Y' (outcome).
    alpha (float): Significance level for the confidence interval.

    Returns:
    Tuple[float, float, float]: ATE, lower bound of the confidence interval, upper bound of the confidence interval.
    """
    treated = data[data['T'] == 1]['Y']
    control = data[data['T'] == 0]['Y']
    
    ate = treated.mean() - control.mean()
    
    # Calculate standard error
    se_treated = treated.std() / np.sqrt(len(treated))
    se_control = control.std() / np.sqrt(len(control))
    se_ate = np.sqrt(se_treated**2 + se_control**2)
    
    # Calculate confidence interval
    z_score = stats.norm.ppf(1 - alpha / 2)
    ci_lower = ate - z_score * se_ate
    ci_upper = ate + z_score * se_ate
    
    return ate, ci_lower, ci_upper

In [17]:
calculate_ate_ci(positive_effect_data, .05)

(np.float64(2.211086786518597),
 np.float64(1.9625239476330625),
 np.float64(2.459649625404131))

In [18]:
calculate_ate_ci(no_effect_data, .05)

(np.float64(0.1829802421329907),
 np.float64(-0.35124762484092764),
 np.float64(0.717208109106909))

In [19]:
def calculate_ate_pvalue(data: pd.DataFrame) -> Tuple[float, float, float]:
    """
    Calculate the ATE and a two-sided normal-approximation p-value for H0: ATE = 0.

    Parameters:
    data (pd.DataFrame): A DataFrame containing columns 'T' (treatment) and 'Y' (outcome).

    Returns:
    Tuple[float, float, float]: ATE, t-statistic, p-value.
    """
    treated = data[data['T'] == 1]['Y']
    control = data[data['T'] == 0]['Y']
    
    ate = treated.mean() - control.mean()
    
    # Standard error of the difference between independent group means.
    se_ate = np.sqrt(
        treated.var(ddof=1) / treated.count()
        + control.var(ddof=1) / control.count()
    )
    t_stat = ate / se_ate

    # sf(x) = 1 - CDF(x), evaluated accurately even for small p-values.
    p_value = 2 * stats.norm.sf(abs(t_stat))
    
    return ate, t_stat, p_value

In [20]:
for name, data in [("Positive effect", positive_effect_data),
                   ("No effect", no_effect_data)]:
    ate, t_stat, p_value = calculate_ate_pvalue(data)
    print(f"{name}: ATE = {ate:.3f}, t = {t_stat:.3f}, p-value = {p_value:.4g}")

Positive effect: ATE = 2.211, t = 17.435, p-value = 4.489e-68
No effect: ATE = 0.183, t = 0.671, p-value = 0.502
